# News Article Scraping

Fetches pre-match news articles from The Guardian API for each match in the dataset.
For each match, articles mentioning the home and away team are retrieved from the 7 days prior to the match date.

The raw articles are cached locally so the API is only hit once. A second pass extracts injury/suspension signals and sentiment scores to produce features that can be merged onto the modelling dataset by `match_api_id`.

In [21]:
import os
import sqlite3
from pathlib import Path
from itertools import cycle

import requests
import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

API_KEYS = [k for k in [
    os.getenv('GUARDIAN_API_KEY_1'),
    os.getenv('GUARDIAN_API_KEY_2'),
    os.getenv('GUARDIAN_API_KEY_3'),
] if k]
assert API_KEYS, 'No GUARDIAN_API_KEY_* keys found — add them to your .env file'
key_pool = cycle(API_KEYS)

GUARDIAN_BASE_URL = 'https://content.guardianapis.com/search'

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DB_PATH   = REPO_ROOT / 'data' / 'database.sqlite'
assert DB_PATH.exists(), f'Missing {DB_PATH}'

print(f'{len(API_KEYS)} API key(s) loaded.')
print(f'Database: {DB_PATH}')

3 API key(s) loaded.
Database: /Users/alexy/CSE/Sports-NLP-Outcome-Predictor/data/database.sqlite


## Load matches and team names from the database

Pull every match alongside both team names so we have the search terms needed for the Guardian API queries.

In [22]:
conn = sqlite3.connect(DB_PATH)

matches = pd.read_sql(
    """
    SELECT
        m.match_api_id,
        m.date,
        m.home_team_api_id,
        m.away_team_api_id,
        ht.team_long_name AS home_team_name,
        at.team_long_name AS away_team_name
    FROM Match m
    JOIN Team ht ON ht.team_api_id = m.home_team_api_id
    JOIN Team at ON at.team_api_id = m.away_team_api_id
    ORDER BY m.date
    """,
    conn,
)
conn.close()

matches['date'] = pd.to_datetime(matches['date'])
print(f'Loaded {len(matches):,} matches')
print(f'Date range: {matches.date.min().date()} → {matches.date.max().date()}')
matches.head()

Loaded 25,979 matches
Date range: 2008-07-18 → 2016-05-25


,match_api_id,date,home_team_api_id,away_team_api_id,home_team_name,away_team_name
0,486263,2008-07-18,10192,9931,BSC Young Boys,FC Basel
1,486264,2008-07-19,9930,10179,FC Aarau,FC Sion
2,486265,2008-07-20,10199,9824,FC Luzern,FC Vaduz
3,486266,2008-07-20,7955,10243,Neuchâtel Xamax,FC Zürich
4,486267,2008-07-23,9931,9956,FC Basel,Grasshopper Club Zürich


## Review team names

Check that the team names from the DB are clean enough to use as Guardian search terms. Flag any that look ambiguous (e.g. "Manchester United" is fine, "Lokomotiv" might return unrelated results).

In [23]:
all_teams = pd.concat([
    matches[['home_team_api_id', 'home_team_name']].rename(columns={'home_team_api_id': 'team_api_id', 'home_team_name': 'team_name'}),
    matches[['away_team_api_id', 'away_team_name']].rename(columns={'away_team_api_id': 'team_api_id', 'away_team_name': 'team_name'}),
]).drop_duplicates().sort_values('team_name').reset_index(drop=True)

print(f'{len(all_teams)} unique teams')
all_teams

299 unique teams


,team_api_id,team_name
0,8350,1. FC Kaiserslautern
1,8722,1. FC Köln
2,8165,1. FC Nürnberg
3,9905,1. FSV Mainz 05
4,8576,AC Ajaccio
...,...,...
294,9868,Xerez Club Deportivo
295,8021,Zagłębie Lubin
296,8027,Zawisza Bydgoszcz
297,4087,Évian Thonon Gaillard FC


## Coverage check

Sample 5 matches per league and measure how many return at least one article. This identifies which leagues have sufficient Guardian coverage before committing to the full fetch.

In [24]:
conn = sqlite3.connect(DB_PATH)
matches_with_league = pd.read_sql(
    """
    SELECT
        m.match_api_id, m.date,
        m.home_team_api_id, m.away_team_api_id,
        ht.team_long_name AS home_team_name,
        at.team_long_name AS away_team_name,
        l.name AS league_name
    FROM Match m
    JOIN Team ht ON ht.team_api_id = m.home_team_api_id
    JOIN Team at ON at.team_api_id = m.away_team_api_id
    JOIN League l ON l.id = m.league_id
    ORDER BY m.date
    """,
    conn,
)
conn.close()
matches_with_league['date'] = pd.to_datetime(matches_with_league['date'])

SAMPLE_PER_LEAGUE = 5
cache = load_cache()
coverage_results = []

leagues = matches_with_league['league_name'].unique()
print(f'Checking coverage for {len(leagues)} leagues ({SAMPLE_PER_LEAGUE} matches each)...\n')

for league in sorted(leagues):
    league_matches = matches_with_league[matches_with_league['league_name'] == league]
    sample_matches = league_matches.sample(min(SAMPLE_PER_LEAGUE, len(league_matches)), random_state=42)

    hits = 0
    total_articles = 0
    for row in sample_matches.itertuples():
        home_arts = fetch_articles(row.home_team_name, row.date, cache)
        away_arts = fetch_articles(row.away_team_name, row.date, cache)
        n = len(home_arts) + len(away_arts)
        if n > 0:
            hits += 1
        total_articles += n

    coverage_results.append({
        'league':                league,
        'matches_sampled':       len(sample_matches),
        'matches_with_coverage': hits,
        'coverage_rate':         hits / len(sample_matches),
        'avg_articles':          total_articles / (len(sample_matches) * 2),
    })

save_cache(cache)

coverage_df = pd.DataFrame(coverage_results).sort_values('coverage_rate', ascending=False).reset_index(drop=True)
print(coverage_df.to_string(index=False))
print(f'\nCache saved: {len(cache)} entries')

Checking coverage for 11 leagues (5 matches each)...

                  league  matches_sampled  matches_with_coverage  coverage_rate  avg_articles
  Belgium Jupiler League                5                      5            1.0           5.0
  England Premier League                5                      5            1.0           9.9
          France Ligue 1                5                      5            1.0           3.8
   Germany 1. Bundesliga                5                      5            1.0           7.1
           Italy Serie A                5                      5            1.0           3.0
  Netherlands Eredivisie                5                      5            1.0           1.2
Portugal Liga ZON Sagres                5                      5            1.0           3.9
         Spain LIGA BBVA                5                      5            1.0           6.7
 Scotland Premier League                5                      4            0.8           2.7
Switze

In [25]:
COVERAGE_THRESHOLD = 0.6
covered_leagues = coverage_df[coverage_df['coverage_rate'] >= COVERAGE_THRESHOLD]['league'].tolist()
print(f'Leagues with coverage >= {COVERAGE_THRESHOLD:.0%}:')
for l in covered_leagues:
    print(f'  {l}')

matches_covered = matches_with_league[matches_with_league['league_name'].isin(covered_leagues)].copy()
print(f'\n{len(matches_covered):,} matches in covered leagues (out of {len(matches_with_league):,} total)')

Leagues with coverage >= 60%:
  Belgium Jupiler League
  England Premier League
  France Ligue 1
  Germany 1. Bundesliga
  Italy Serie A
  Netherlands Eredivisie
  Portugal Liga ZON Sagres
  Spain LIGA BBVA
  Scotland Premier League
  Switzerland Super League

24,059 matches in covered leagues (out of 25,979 total)


## Article fetching pipeline

Fetch up to 10 articles per team per match from the Guardian `football` section in the 7 days before the match date. Results are cached to `data/articles_cache.json` so the API is only hit once per (team, date_window) pair.

In [26]:
import json
import time
import unicodedata
from datetime import timedelta

CACHE_PATH       = REPO_ROOT / 'data' / 'articles_cache.json'
LOOKBACK_DAYS    = 7
RATE_LIMIT_DELAY = 0.15

def load_cache():
    if CACHE_PATH.exists():
        with open(CACHE_PATH) as f:
            return json.load(f)
    return {}

def save_cache(cache):
    tmp = CACHE_PATH.with_suffix('.tmp')
    with open(tmp, 'w') as f:
        json.dump(cache, f)
    tmp.rename(CACHE_PATH)

def clean_team_name(name):
    return unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('ascii').strip()

def fetch_articles(team_name, match_date, cache):
    from_date  = (match_date - timedelta(days=LOOKBACK_DAYS)).strftime('%Y-%m-%d')
    to_date    = (match_date - timedelta(days=1)).strftime('%Y-%m-%d')
    cache_key  = f"{team_name}|{from_date}|{to_date}"

    if cache_key in cache:
        return cache[cache_key]

    try:
        response = requests.get(
            GUARDIAN_BASE_URL,
            params={
                'q':           clean_team_name(team_name),
                'section':     'football',
                'from-date':   from_date,
                'to-date':     to_date,
                'show-fields': 'headline,bodyText',
                'page-size':   10,
                'api-key':     next(key_pool),
            },
            timeout=10,
        )
        response.raise_for_status()
        results = response.json()['response']['results']
        articles = [
            {
                'headline': r['fields'].get('headline', ''),
                'body':     r['fields'].get('bodyText', ''),
                'date':     r['webPublicationDate'],
            }
            for r in results
        ]
    except Exception as e:
        print(f'  ERROR fetching {team_name} {from_date}: {e}')
        articles = []

    cache[cache_key] = articles
    time.sleep(RATE_LIMIT_DELAY)
    return articles

print('Fetch function ready.')

Fetch function ready.


## Run the fetch loop

Iterates over every match, fetches articles for the home and away team, and saves progress to the cache after every 100 matches. Re-running this cell will skip already-cached entries and only fetch what's missing.

In [ ]:
cache = load_cache()
print(f'Cache loaded: {len(cache)} existing entries')

# Deduplicate by unique (team, window) pairs before fetching ---
# Many matches share the same team/week, so we pre-compute all unique pairs
# and fetch each only once. The match loop below then reads entirely from cache.
unique_pairs = set()
for row in matches_covered.itertuples():
    for team in (row.home_team_name, row.away_team_name):
        from_date = (row.date - timedelta(days=LOOKBACK_DAYS)).strftime('%Y-%m-%d')
        to_date   = (row.date - timedelta(days=1)).strftime('%Y-%m-%d')
        unique_pairs.add((team, row.date, from_date, to_date))

already_cached = sum(1 for team, _, fd, td in unique_pairs if f"{team}|{fd}|{td}" in cache)
to_fetch = len(unique_pairs) - already_cached
print(f'Unique (team, window) pairs: {len(unique_pairs):,}')
print(f'Already cached: {already_cached:,}  |  To fetch: {to_fetch:,}')
print(f'Estimated time: ~{to_fetch * RATE_LIMIT_DELAY / 60:.0f} min across {len(API_KEYS)} key(s)\n')

for i, (team, date, _, _) in enumerate(sorted(unique_pairs), 1):
    fetch_articles(team, date, cache)
    if i % 500 == 0:
        save_cache(cache)
        print(f'  {i}/{len(unique_pairs)} pairs fetched — cache size: {len(cache)} entries')

save_cache(cache)
print(f'\nAll unique pairs fetched. Cache size: {len(cache)} entries.')

# --- Build records from cache (no API calls) ---
records = []
for row in matches_covered.itertuples():
    home_articles = fetch_articles(row.home_team_name, row.date, cache)
    away_articles = fetch_articles(row.away_team_name, row.date, cache)
    records.append({
        'match_api_id':       row.match_api_id,
        'home_article_count': len(home_articles),
        'away_article_count': len(away_articles),
        'home_articles':      home_articles,
        'away_articles':      away_articles,
    })

print(f'Records built for {len(records):,} matches.')
print(f'Cache saved to {CACHE_PATH}')

Cache loaded: 310 existing entries
Unique (team, window) pairs: 48,118
Already cached: 300  |  To fetch: 47,818
Estimated time: ~120 min across 3 key(s)



KeyboardInterrupt: 

## Sanity check

Inspect a sample match to confirm articles are being retrieved correctly.

In [ ]:
sample = next(r for r in records if r['home_article_count'] > 0)
match_row = matches[matches['match_api_id'] == sample['match_api_id']].iloc[0]

print(f"Match: {match_row['home_team_name']} vs {match_row['away_team_name']} on {match_row['date'].date()}")
print(f"Home articles: {sample['home_article_count']}  |  Away articles: {sample['away_article_count']}\n")

print('--- Home team article headlines ---')
for a in sample['home_articles']:
    print(f"  {a['date'][:10]} — {a['headline']}")